# 12 — Service layer (`factory_floor/services.py`)

Phase 2 of the professionalization work extracted the business logic of a diagnostic turn out of `app.py` into `factory_floor/services.py` — plain functions, no Streamlit. This notebook runs a turn through that layer end to end (blocking **and** streaming) against the real vector store, with `assert` checks, so the `nbconvert` sweep catches any regression in the extraction.

It mirrors notebook 07, but calls `services.run_diagnostic()` instead of `agent.run_diagnostic_agent()` directly.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor import services
from factory_floor.config import COLLECTION_NAME, VECTOR_DIR
from factory_floor.vectorstore import get_embeddings, load_vectorstore

vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=get_embeddings())
print('vector store loaded:', vectorstore._collection.count(), 'chunks')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vector store loaded: 12301 chunks


## Blocking turn

A real VFD fault-code question, GENERAL machine (no history tool). The agent should call `search_manuals` and cite a real page.

In [2]:
req = services.DiagnosticRequest(
    question_text='F30021 ground fault after several hours of running — what should be checked?',
    machine_id='GENERAL',
    equipment_type='VFD',
    language='English',
)
result = services.run_diagnostic(req, vectorstore=vectorstore)

print(result.answer[:800])
print('\n--- tools used ---')
for entry in (result.tool_trace or []):
    print(entry['tool'], entry.get('input'))
print('\nrun_id:', result.run_id)

Safety precautions:
- Before inspecting or working on the drive or motor, isolate and de-energize the equipment.
- Apply lockout/tagout procedures to ensure the equipment cannot be accidentally energized.
- Wait for the DC link capacitors to discharge fully before touching any terminals.
- Verify absence of voltage with appropriate testing equipment.
- Only qualified personnel should perform these checks.

Regarding the F30021 ground fault after several hours of running, the manuals indicate the following checks should be performed:

1. Check the power cable connections for any loose or damaged connections.
2. Inspect the power cables for any short-circuits or ground faults.
3. Check the motor itself for any ground faults.
4. Verify the motor circuit configuration (e.g., star-delta) is cor

--- tools used ---
search_manuals {'query': 'F30021 ground fault'}

run_id: ac242c70-5833-4ddf-a1d8-b4c3da355dc8


In [3]:
assert isinstance(result, services.DiagnosticResult)
assert result.answer and len(result.answer) > 50
assert result.run_id
assert result.blocked is False and result.cache_hit is False
assert any(e['tool'] == 'search_manuals' for e in (result.tool_trace or [])), 'expected a manual search for a fault code'
assert result.documents, 'expected retrieved documents'
print('blocking turn OK')

blocking turn OK


## Streaming turn

Same request, `stream=True`. The result object stays empty until the generator is fully consumed (exactly how `app.py` drives it via `st.write_stream`).

In [4]:
generator, streamed_result = services.run_diagnostic(req, vectorstore=vectorstore, stream=True)
assert streamed_result.answer is None, 'result must be empty before the generator is consumed'

chunks = list(generator)
streamed_text = ''.join(chunks)
print(streamed_text[:800])

Safety precautions:
- Before inspecting or working on the drive or motor, isolate and de-energize the equipment.
- Apply lockout/tagout procedures to ensure the equipment cannot be accidentally energized.
- Wait for the DC link capacitors to discharge fully before touching any terminals.
- Verify absence of voltage with appropriate testing equipment.
- Only qualified personnel should perform these checks.

Regarding the F30021 ground fault after several hours of running, the manuals indicate the following should be checked:

1. Power cable connections for any loose or damaged connections.
2. Power cables themselves for any short-circuits or ground faults.
3. Motor circuit configuration (e.g., star-delta wiring).
4. Motor condition for any ground faults.
5. Current transformer (CT) for defe


In [5]:
assert streamed_result.answer, 'answer should be populated after consuming the generator'
assert streamed_result.answer.strip() == streamed_text.strip() or streamed_text in streamed_result.answer
assert streamed_result.run_id
print('streaming turn OK')

streaming turn OK


## Turn assembly

The dict `app.py` appends to its conversation list.

In [6]:
turn = services.assemble_turn(
    streamed_result, image_bytes=None, classification=None, vision_context=None, language='English'
)
assert set(turn) == {
    'type', 'question', 'answer', 'documents', 'sources', 'tool_trace', 'safety',
    'image_bytes', 'vision_context', 'predicted_label', 'is_defective', 'language',
}
assert turn['type'] == 'agent'
assert turn['answer'] == streamed_result.answer
turn['question'], turn['language'], turn['safety']['action'], len(turn['answer'])

('F30021 ground fault after several hours of running — what should be checked?',
 'English',
 'pass',
 1208)

## Typo pre-check

`services.check_typo` is the pre-API guard `app.py` runs before submitting.

In [7]:
assert services.check_typo('display reads F3OO21') == [('F3OO21', 'F30021')]
assert services.check_typo('the motor is overheating') == []
print('typo pre-check OK')

typo pre-check OK
